# AIC 2026 — Local Hybrid Search Engine

- `EngineConfig`, các dataclass tài liệu và báo cáo;
- `VectorIndex`, `OpenClipTextEncoder`;
- `LocalHybridSearchEngine.search()` và
  `LocalHybridSearchEngine.search_sequence()`;
- các hàm build SQLite FTS5, FAISS/NumPy và weighted RRF.

Input bắt buộc có thể là thư mục hoặc ZIP:

1. `01_keyframes_output.zip`
2. `02_ocr_output.zip`
3. `03_asr_output.zip`
4. `04_scene_clips_output.zip`

Output 05 (semantic), 06 (validation) và 07 (metadata) là tùy chọn; nếu có,
loader sẽ tự nạp. Chạy các cell từ trên xuống.

## 1. Cài thư viện

`numpy` là bắt buộc. `faiss-cpu` được cài mặc định để tìm vector nhanh; nếu
không có FAISS, engine vẫn tự chuyển sang NumPy. `open_clip_torch` chỉ cần khi
muốn nhập một mô tả thị giác mới và encode nó thành vector ngay trong notebook.

In [74]:
import importlib.util
import os
import subprocess
import sys

SKIP_INSTALL = os.environ.get("AIC_SKIP_INSTALL", "0") == "1"
INSTALL_FAISS = os.environ.get("AIC_INSTALL_FAISS", "1") == "1"
INSTALL_OPENCLIP = os.environ.get("AIC_INSTALL_OPENCLIP", "0") == "1"

packages = []
if importlib.util.find_spec("numpy") is None:
    packages.append("numpy>=1.24,<3")
if INSTALL_FAISS and importlib.util.find_spec("faiss") is None:
    packages.append("faiss-cpu>=1.8")
if INSTALL_OPENCLIP and importlib.util.find_spec("open_clip") is None:
    packages.append("open_clip_torch>=2.24")

if packages and not SKIP_INSTALL:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages]
    )
    print("Đã cài:", ", ".join(packages))
elif SKIP_INSTALL:
    print("Bỏ qua cài thư viện vì AIC_SKIP_INSTALL=1.")
else:
    print("Các thư viện đã sẵn sàng.")

print("Python:", sys.version.split()[0])

Các thư viện đã sẵn sàng.
Python: 3.12.13


## 2. Định nghĩa cấu hình và kiểu dữ liệu

Các class trong phần này được giữ nguyên từ file Python một-file. Chúng là
schema chung đi xuyên suốt pha build và pha search.

In [75]:
from __future__ import annotations

__version__ = "0.2.1-notebook"

# ===========================================================================
# Logic preserved from: config.py
# ===========================================================================
from dataclasses import asdict, dataclass
from typing import Literal

@dataclass(slots=True)
class EngineConfig:
    """Search and index settings kept in ``index_manifest.json``."""
    vector_backend: Literal['auto', 'faiss', 'numpy'] = 'auto'
    hnsw_m: int = 32
    hnsw_ef_construction: int = 200
    hnsw_ef_search: int = 64
    rrf_k: int = 60
    lexical_weight: float = 1.0
    vector_weight: float = 1.0
    lexical_candidates: int = 100
    vector_candidates: int = 100
    min_lexical_terms: int = 2
    min_lexical_coverage: float = 0.34
    min_scene_vector_similarity: float = 0.2
    min_frame_vector_similarity: float = 0.2
    require_visual_query_for_vector: bool = True
    semantic_weight: float = 1.35
    ocr_weight: float = 1.0
    speech_weight: float = 1.0
    tags_weight: float = 1.1
    event_weight: float = 0.9
    scene_vector_weight: float = 1.35
    frame_vector_weight: float = 1.0
    needs_review_penalty: float = 0.75
    exclude_invalid: bool = True
    keep_numpy_fallback: bool = False

    def validate(self) -> None:
        if self.vector_backend not in {'auto', 'faiss', 'numpy'}:
            raise ValueError(f'Unsupported vector backend: {self.vector_backend}')
        if self.hnsw_m <= 0 or self.hnsw_ef_construction <= 0 or self.hnsw_ef_search <= 0:
            raise ValueError('HNSW parameters must be positive')
        if self.rrf_k < 0:
            raise ValueError('rrf_k must be non-negative')
        if self.lexical_candidates <= 0 or self.vector_candidates <= 0:
            raise ValueError('Candidate counts must be positive')
        if self.min_lexical_terms <= 0:
            raise ValueError('min_lexical_terms must be positive')
        if not 0.0 <= self.min_lexical_coverage <= 1.0:
            raise ValueError('min_lexical_coverage must be in [0, 1]')
        if not -1.0 <= self.min_scene_vector_similarity <= 1.0:
            raise ValueError('min_scene_vector_similarity must be in [-1, 1]')
        if not -1.0 <= self.min_frame_vector_similarity <= 1.0:
            raise ValueError('min_frame_vector_similarity must be in [-1, 1]')
        if not 0.0 <= self.needs_review_penalty <= 1.0:
            raise ValueError('needs_review_penalty must be in [0, 1]')

    def to_dict(self) -> dict:
        self.validate()
        return asdict(self)

    @classmethod
    def from_dict(cls, value: dict) -> 'EngineConfig':
        known = cls.__dataclass_fields__
        return cls(**{key: val for key, val in value.items() if key in known})

# ===========================================================================
# Logic preserved from: records.py
# ===========================================================================
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

@dataclass(slots=True)
class SceneDocument:
    scene_id: str
    video_id: str
    scene_no: int
    start_sec: float
    end_sec: float
    clip_path: str
    start_frame: int = 0
    end_frame: int = 0
    representative_keyframe_id: str = ''
    vector_row: int = -1
    ocr_text: str = ''
    transcript: str = ''
    caption_vi: str = ''
    caption_en: str = ''
    speech_summary: str = ''
    scene_type: str = 'other'
    visible_text: str = ''
    keywords: str = ''
    entities: str = ''
    actions: str = ''
    attributes: str = ''
    relations: str = ''
    event_text: str = ''
    temporal_events: list[dict[str, Any]] = field(default_factory=list)
    semantic_status: str = 'missing'
    quality_status: str = 'passed'
    quality_penalty: float = 1.0
    quality_errors: list[str] = field(default_factory=list)
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(slots=True)
class KeyframeDocument:
    keyframe_id: str
    scene_id: str
    frame_idx: int
    timestamp_sec: float
    image_path: str
    vector_row: int
    quality_score: float = 0.0
    ocr_text: str = ''
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(slots=True)
class LoadedComponents:
    scenes: list[SceneDocument]
    keyframes: list[KeyframeDocument]
    scene_embeddings: Any
    keyframe_embeddings: Any | None
    scene_embedding_model: str
    keyframe_embedding_model: str
    embedding_dimension: int
    source_root: Path
    stats: dict[str, Any]
    warnings: list[str]

@dataclass(slots=True)
class BuildReport:
    index_dir: str
    database_path: str
    vector_backend: str
    embedding_dimension: int
    scene_count: int
    keyframe_count: int
    video_count: int
    warnings: list[str]
    elapsed_sec: float

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

## 3. Utility: chuẩn hóa query, độ phủ từ khóa và giải nén ZIP an toàn

In [76]:
from __future__ import annotations

# ===========================================================================
# Logic preserved from: utils.py
# ===========================================================================
import json
import re
import unicodedata
import zipfile
from pathlib import Path
from typing import Iterable, Iterator
SCENE_RE = re.compile('^(?P<video>.+)_S(?P<number>\\d+)$')
VIETNAMESE_QUERY_STOPWORDS = frozenset({'ai', 'bị', 'bởi', 'các', 'cái', 'cho', 'chỉ', 'có', 'của', 'cũng', 'đã', 'đang', 'đến', 'để', 'đó', 'được', 'gì', 'khi', 'không', 'là', 'lại', 'lúc', 'mà', 'một', 'này', 'những', 'ở', 'sau', 'sẽ', 'theo', 'thì', 'trên', 'trong', 'trước', 'từ', 'và', 'vào', 'về', 'với'})

def read_jsonl(path: Path) -> Iterator[dict]:
    with path.open('r', encoding='utf-8-sig') as handle:
        for line_no, line in enumerate(handle, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSONL at {path}:{line_no}: {exc}') from exc

def scene_parts(scene_id: str) -> tuple[str, int]:
    match = SCENE_RE.match(scene_id)
    if not match:
        raise ValueError(f'scene_id does not follow <video_id>_S####: {scene_id}')
    return (match.group('video'), int(match.group('number')))

def normalize_space(value: object) -> str:
    return ' '.join(str(value or '').split())

def accent_fold(value: str) -> str:
    decomposed = unicodedata.normalize('NFD', value)
    return ''.join((ch for ch in decomposed if unicodedata.category(ch) != 'Mn')).replace('đ', 'd').replace('Đ', 'D')

def query_terms(text: str) -> list[str]:
    """Return unique content-bearing terms used by lexical retrieval."""
    folded_stopwords = {accent_fold(token).casefold() for token in VIETNAMESE_QUERY_STOPWORDS}
    raw_tokens = re.findall('[^\\W_]+', accent_fold(normalize_space(text)).casefold(), flags=re.UNICODE)
    return list(dict.fromkeys((token for token in raw_tokens if token not in folded_stopwords and (len(token) >= 2 or token.isdigit()))))

def lexical_coverage(query: str, evidence: str) -> tuple[float, list[str]]:
    """Measure how many distinct query terms occur in one evidence field."""
    terms = query_terms(query)
    if not terms:
        return (0.0, [])
    evidence_terms = set(query_terms(evidence))
    matched = [term for term in terms if term in evidence_terms]
    return (len(matched) / len(terms), matched)

def make_fts_query(text: str, match_all: bool=False) -> str:
    """Turn natural text into a safe FTS5 query.

    FTS5 operators from user text are intentionally discarded. ``OR`` gives
    retrieval-oriented recall; ``match_all=True`` is useful for strict filters.
    """
    terms = query_terms(text)
    if not terms:
        return ''
    operator = ' AND ' if match_all else ' OR '
    return operator.join((f'"{term.replace(chr(34), chr(34) * 2)}"' for term in terms))

def safe_extract_zip(archive: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(archive) as zipped:
        for info in zipped.infolist():
            target = (destination / info.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f'Unsafe path in {archive}: {info.filename}')
        zipped.extractall(destination)

def unique_paths(paths: Iterable[Path]) -> list[Path]:
    seen: set[str] = set()
    result: list[Path] = []
    for path in paths:
        resolved = str(path.resolve())
        if resolved not in seen:
            seen.add(resolved)
            result.append(path.resolve())
    return sorted(result)

## 4. Ingestion

Luồng chính:

`load_components()` → `_prepare_roots()` → các loader Output 01–07 →
`SceneDocument`/`KeyframeDocument`.

Nếu Output 07 không tồn tại, scene embedding được tạo bằng trung bình các
keyframe embedding của scene. ASR timestamp được căn lại theo độ giao thực với
scene để hạn chế lời nói bị lan qua ranh giới scene.

In [77]:
from __future__ import annotations

# ===========================================================================
# Logic preserved from: ingest.py
# ===========================================================================
import csv
import json
import re
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Any, Iterable
import numpy as np
RELEVANT_NAMES = {'scene_metadata.jsonl', 'scene_embeddings.npy', 'scene_docs.jsonl', 'scene_visual_embeddings.npy', 'frame_docs.jsonl', 'frame_visual_embeddings.npy', 'metadata_manifest.json', 'validation_report.json', 'scene_semantics_qwen3vl.jsonl', 'caption_scenes_qwen3vl.jsonl', 'scene_captions_selfhosted.jsonl', 'scene_text_index_ready_selfhosted.jsonl', 'keyframe_index.csv', 'keyframe_visual_embeddings.npy', 'keyframes.json', 'ocr_keyframes.jsonl', 'ocr_scenes.jsonl', 'asr_scenes.jsonl', 'asr_segments.json', 'asr_segments.jsonl', 'scene_clip_manifest.json', 'component_validation_report.json'}
SCENE_ID_IN_TEXT = re.compile('(?P<scene_id>[A-Za-z0-9_.-]+_S\\d+)')

def _canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))

def _discover(roots: list[Path], filename: str) -> list[Path]:
    return unique_paths((path for root in roots for path in root.rglob(filename) if path.is_file()))

def _prepare_roots(input_root: Path, staging_dir: Path) -> list[Path]:
    input_root = input_root.resolve()
    if not input_root.exists():
        raise FileNotFoundError(f'Input root does not exist: {input_root}')
    roots = [input_root]
    staging_dir.mkdir(parents=True, exist_ok=True)
    for number, archive in enumerate(sorted(input_root.rglob('*.zip'))):
        try:
            with zipfile.ZipFile(archive) as zipped:
                names = {Path(name).name for name in zipped.namelist()}
        except zipfile.BadZipFile:
            continue
        if not names.intersection(RELEVANT_NAMES) and (not any(('valid' in name.casefold() or 'quality_report' in name.casefold() for name in names))):
            continue
        destination = staging_dir / f'{number:04d}_{archive.stem}'
        if destination.exists():
            shutil.rmtree(destination)
        destination.mkdir(parents=True)
        safe_extract_zip(archive, destination)
        roots.append(destination)
    return roots

def _read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8-sig'))

def _put_record(target: dict[str, dict], key: str, value: dict, source: Path) -> bool:
    if key not in target:
        target[key] = value
        return True
    if _canonical(target[key]) == _canonical(value):
        return False
    raise ValueError(f'Duplicate id {key!r} with different content in {source}')

def _model_name(info: dict | None) -> str:
    if not info:
        return 'open_clip:ViT-B-32:openai'
    if info.get('model'):
        return str(info['model'])
    sources = info.get('source_embedding_models') or []
    for source in sources:
        if isinstance(source, dict) and source.get('model'):
            return str(source['model'])
    return 'open_clip:ViT-B-32:openai'

def _metadata_manifest_model(info: dict | None) -> str:
    if not info:
        return 'open_clip:ViT-B-32:openai'
    models = (info.get('embedding') or {}).get('component_models') or []
    return _model_name(models[0]) if models else 'open_clip:ViT-B-32:openai'

def _load_keyframes(roots: list[Path]) -> tuple[list[KeyframeDocument], np.ndarray | None, str]:
    documents: list[KeyframeDocument] = []
    vectors: list[np.ndarray] = []
    seen: dict[str, tuple[dict, np.ndarray]] = {}
    model_names: set[str] = set()
    dimension: int | None = None
    for index_path in _discover(roots, 'keyframe_index.csv'):
        matrix_path = index_path.parent / 'keyframe_visual_embeddings.npy'
        if not matrix_path.exists():
            raise FileNotFoundError(f'Missing keyframe vectors next to {index_path}')
        matrix = np.asarray(np.load(matrix_path, allow_pickle=False), dtype=np.float32)
        if matrix.ndim != 2:
            raise ValueError(f'Keyframe embeddings must be 2D: {matrix_path}')
        dimension = dimension or int(matrix.shape[1])
        if matrix.shape[1] != dimension:
            raise ValueError(f'Mixed keyframe embedding dimensions: {matrix_path}')
        with index_path.open('r', encoding='utf-8-sig', newline='') as handle:
            rows = list(csv.DictReader(handle))
        if len(rows) != len(matrix):
            raise ValueError(f'CSV/vector row mismatch in {index_path}')
        quality_path = index_path.parent / 'keyframes.json'
        quality_items = _read_json(quality_path) if quality_path.exists() else []
        quality_by_id = {str(item['keyframe_id']): item for item in quality_items}
        model_path = index_path.parent / 'model_info.json'
        if model_path.exists():
            model_names.add(_model_name(_read_json(model_path)))
        for csv_position, row in enumerate(rows):
            keyframe_id = str(row['keyframe_id'])
            local_row = int(row.get('embedding_row', csv_position))
            if not 0 <= local_row < len(matrix):
                raise IndexError(f'{keyframe_id}: invalid embedding_row={local_row}')
            vector = np.asarray(matrix[local_row], dtype=np.float32)
            public = {'keyframe_id': keyframe_id, 'scene_id': str(row['scene_id']), 'frame_idx': int(row['frame_idx']), 'timestamp_sec': float(row['timestamp_sec']), 'image_path': str(row.get('image_path', ''))}
            if keyframe_id in seen:
                old_public, old_vector = seen[keyframe_id]
                if old_public != public or not np.allclose(old_vector, vector, atol=1e-06):
                    raise ValueError(f'Duplicate keyframe differs: {keyframe_id}')
                continue
            seen[keyframe_id] = (public, vector)
            item = quality_by_id.get(keyframe_id, {})
            quality = item.get('quality', {}) if isinstance(item, dict) else {}
            documents.append(KeyframeDocument(**public, vector_row=len(vectors), quality_score=float(quality.get('score', 0.0) or 0.0), metadata=item))
            vectors.append(vector)
    if not vectors:
        for docs_path in _discover(roots, 'frame_docs.jsonl'):
            matrix_path = docs_path.parent / 'frame_visual_embeddings.npy'
            if not matrix_path.exists():
                raise FileNotFoundError(f'Missing frame vectors next to {docs_path}')
            matrix = np.asarray(np.load(matrix_path, allow_pickle=False), dtype=np.float32)
            items = list(read_jsonl(docs_path))
            if matrix.ndim != 2 or len(matrix) != len(items):
                raise ValueError(f'Frame docs/vector mismatch: {docs_path}')
            dimension = dimension or int(matrix.shape[1])
            if matrix.shape[1] != dimension:
                raise ValueError(f'Mixed frame embedding dimensions: {matrix_path}')
            manifest_path = docs_path.parent / 'metadata_manifest.json'
            if manifest_path.exists():
                model_names.add(_metadata_manifest_model(_read_json(manifest_path)))
            for position, item in enumerate(items):
                keyframe_id = str(item['keyframe_id'])
                local_row = int(item.get('visual_embedding_row', item.get('embedding_row', position)))
                if not 0 <= local_row < len(matrix):
                    raise IndexError(f'{keyframe_id}: invalid visual_embedding_row={local_row}')
                vector = np.asarray(matrix[local_row], dtype=np.float32)
                public = {'keyframe_id': keyframe_id, 'scene_id': str(item['scene_id']), 'frame_idx': int(item['frame_idx']), 'timestamp_sec': float(item['timestamp_sec']), 'image_path': str(item.get('image_path') or item.get('image_asset_uri', ''))}
                if keyframe_id in seen:
                    old_public, old_vector = seen[keyframe_id]
                    if old_public != public or not np.allclose(old_vector, vector, atol=1e-06):
                        raise ValueError(f'Duplicate rich keyframe differs: {keyframe_id}')
                    continue
                seen[keyframe_id] = (public, vector)
                quality = item.get('quality') if isinstance(item.get('quality'), dict) else {}
                documents.append(KeyframeDocument(**public, vector_row=len(vectors), quality_score=float(quality.get('score', 0.0) or 0.0), ocr_text=normalize_space(item.get('ocr_text', '')), metadata={'quality': quality, 'selection': item.get('selection', {}), 'ocr_status': item.get('ocr_status', ''), 'provenance': item.get('provenance', {})}))
                vectors.append(vector)
    if not vectors:
        return ([], None, 'open_clip:ViT-B-32:openai')
    if len(model_names) > 1:
        raise ValueError(f'Mixed keyframe embedding models: {sorted(model_names)}')
    matrix = np.ascontiguousarray(np.stack(vectors), dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if np.any(~np.isfinite(norms)) or np.any(norms < 1e-08):
        raise ValueError('Invalid keyframe vectors')
    matrix /= norms
    return (documents, matrix, next(iter(model_names), 'open_clip:ViT-B-32:openai'))

def _load_scene_text(roots: list[Path], filename: str, key_field: str='scene_id') -> dict[str, dict]:
    result: dict[str, dict] = {}
    for path in _discover(roots, filename):
        for item in read_jsonl(path):
            _put_record(result, str(item[key_field]), item, path)
    return result

def _load_asr_segments(roots: list[Path]) -> list[dict]:
    """Load timestamped ASR once so long segments can be aligned safely."""
    output: dict[str, dict] = {}

    def add(item: dict, key: str, source: Path) -> None:
        previous = output.get(key)
        if previous is not None:
            comparable = lambda value: (str(value.get('video_id', '')), float(value.get('start_sec', 0.0) or 0.0), float(value.get('end_sec', 0.0) or 0.0), normalize_space(value.get('text', '')))
            if comparable(previous) != comparable(item):
                raise ValueError(f'Duplicate ASR segment {key!r} differs in {source}')
            output[key] = {**previous, **item}
            return
        output[key] = item
    for path in _discover(roots, 'asr_segments.json'):
        value = _read_json(path)
        if not isinstance(value, dict):
            continue
        video_id = str((value.get('scope') or {}).get('video_id', ''))
        for position, raw in enumerate(value.get('segments') or [], 1):
            if not isinstance(raw, dict):
                continue
            item = dict(raw)
            item.setdefault('video_id', video_id)
            key = str(item.get('segment_id') or f'{video_id}:{position}')
            add(item, key, path)
    for path in _discover(roots, 'asr_segments.jsonl'):
        for position, raw in enumerate(read_jsonl(path), 1):
            item = dict(raw)
            key = str(item.get('segment_id') or f"{item.get('video_id', '')}:{position}")
            add(item, key, path)
    return list(output.values())

def _align_asr_segments(metadata: dict[str, dict], segments: list[dict]) -> dict[str, str]:
    """Assign ASR to scenes with material overlap, not every boundary touch."""
    scenes_by_video: dict[str, list[tuple[str, float, float]]] = defaultdict(list)
    for scene_id, item in metadata.items():
        video_id = str(item.get('video_id') or scene_parts(scene_id)[0])
        scenes_by_video[video_id].append((scene_id, float(item['start_sec']), float(item['end_sec'])))
    for values in scenes_by_video.values():
        values.sort(key=lambda row: (row[1], row[2], row[0]))
    assigned: dict[str, list[str]] = defaultdict(list)
    for segment in segments:
        text = normalize_space(segment.get('text', ''))
        video_id = str(segment.get('video_id', ''))
        candidates = scenes_by_video.get(video_id, [])
        if not text or not candidates:
            continue
        start = float(segment.get('start_sec', 0.0) or 0.0)
        end = max(start, float(segment.get('end_sec', start) or start))
        duration = max(0.0, end - start)
        minimum_overlap = min(0.75, max(0.05, duration * 0.1))
        selected: list[str] = []
        for scene_id, scene_start, scene_end in candidates:
            overlap = max(0.0, min(end, scene_end) - max(start, scene_start))
            if overlap + 1e-09 >= minimum_overlap:
                selected.append(scene_id)
        if not selected:
            midpoint = (start + end) / 2.0
            containing = [scene_id for scene_id, scene_start, scene_end in candidates if scene_start <= midpoint <= scene_end]
            selected = containing[:1]
        for scene_id in selected:
            if text not in assigned[scene_id]:
                assigned[scene_id].append(text)
    return {scene_id: normalize_space(' '.join(texts)) for scene_id, texts in assigned.items()}

def _load_scene_manifests(roots: list[Path]) -> dict[str, dict]:
    result: dict[str, dict] = {}
    for path in _discover(roots, 'scene_clip_manifest.json'):
        raw = _read_json(path)
        if not isinstance(raw, list):
            raise ValueError(f'Expected manifest list: {path}')
        for item in raw:
            _put_record(result, str(item['scene_id']), dict(item), path)
    return result

def _load_metadata(roots: list[Path]) -> tuple[dict[str, dict], dict[str, np.ndarray], str]:
    records: dict[str, dict] = {}
    vectors: dict[str, np.ndarray] = {}
    models: set[str] = set()
    dimension: int | None = None
    for path in _discover(roots, 'scene_metadata.jsonl'):
        matrix_path = path.parent / 'scene_embeddings.npy'
        if not matrix_path.exists():
            raise FileNotFoundError(f'Missing scene_embeddings.npy next to {path}')
        matrix = np.asarray(np.load(matrix_path, allow_pickle=False), dtype=np.float32)
        items = list(read_jsonl(path))
        if matrix.ndim != 2 or len(matrix) != len(items):
            raise ValueError(f'Metadata/vector mismatch: {path}')
        dimension = dimension or int(matrix.shape[1])
        if matrix.shape[1] != dimension:
            raise ValueError(f'Mixed scene embedding dimensions: {matrix_path}')
        model_path = path.parent / 'model_info.json'
        if model_path.exists():
            models.add(_model_name(_read_json(model_path)))
        for item in items:
            scene_id = str(item['scene_id'])
            row = int(item['embedding_row'])
            if not 0 <= row < len(matrix):
                raise IndexError(f'{scene_id}: invalid scene embedding_row={row}')
            vector = np.asarray(matrix[row], dtype=np.float32)
            inserted = _put_record(records, scene_id, item, path)
            if inserted:
                vectors[scene_id] = vector
            elif not np.allclose(vectors[scene_id], vector, atol=1e-06):
                raise ValueError(f'Duplicate scene vector differs: {scene_id}')
    if not records:
        for path in _discover(roots, 'scene_docs.jsonl'):
            matrix_path = path.parent / 'scene_visual_embeddings.npy'
            if not matrix_path.exists():
                raise FileNotFoundError(f'Missing scene vectors next to {path}')
            matrix = np.asarray(np.load(matrix_path, allow_pickle=False), dtype=np.float32)
            items = list(read_jsonl(path))
            if matrix.ndim != 2 or len(matrix) != len(items):
                raise ValueError(f'Scene docs/vector mismatch: {path}')
            dimension = dimension or int(matrix.shape[1])
            if matrix.shape[1] != dimension:
                raise ValueError(f'Mixed scene embedding dimensions: {matrix_path}')
            manifest_path = path.parent / 'metadata_manifest.json'
            if manifest_path.exists():
                models.add(_metadata_manifest_model(_read_json(manifest_path)))
            for position, raw in enumerate(items):
                item = dict(raw)
                scene_id = str(item['scene_id'])
                row = int(item.get('visual_embedding_row', item.get('embedding_row', position)))
                if not 0 <= row < len(matrix):
                    raise IndexError(f'{scene_id}: invalid visual_embedding_row={row}')
                item['embedding_row'] = row
                item.setdefault('keyframes', [])
                inserted = _put_record(records, scene_id, item, path)
                vector = np.asarray(matrix[row], dtype=np.float32)
                if inserted:
                    vectors[scene_id] = vector
                elif not np.allclose(vectors[scene_id], vector, atol=1e-06):
                    raise ValueError(f'Duplicate rich scene vector differs: {scene_id}')
    if len(models) > 1:
        raise ValueError(f'Mixed scene embedding models: {sorted(models)}')
    return (records, vectors, next(iter(models), 'open_clip:ViT-B-32:openai'))

def _valid_scene_key(item: dict) -> str:
    for key in ('scene_key', 'scene_id'):
        value = str(item.get(key, ''))
        if SCENE_ID_IN_TEXT.fullmatch(value):
            return value
    return ''

def _flatten_item(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, str):
        return normalize_space(value)
    if isinstance(value, dict):
        ordered_keys = ('text', 'name', 'label', 'type', 'description', 'subject', 'predicate', 'relation', 'object', 'action', 'attribute')
        parts = [_flatten_item(value.get(key)) for key in ordered_keys if key in value]
        return normalize_space(' '.join((part for part in parts if part)))
    if isinstance(value, list):
        return normalize_space(' '.join(filter(None, (_flatten_item(item) for item in value))))
    return normalize_space(value)

def _first_text(mapping: dict, keys: tuple[str, ...]) -> str:
    for key in keys:
        text = _flatten_item(mapping.get(key))
        if text:
            return text
    return ''

def _adapt_selfhosted_semantic(item: dict) -> dict | None:
    scene_id = _valid_scene_key(item)
    if not scene_id:
        return None
    if isinstance(item.get('semantic'), dict):
        output = dict(item)
        output['scene_id'] = scene_id
        return output
    context = item.get('scene_context')
    context = context if isinstance(context, dict) else {}
    parsed_frames = [frame.get('parsed') for frame in item.get('keyframes') or [] if isinstance(frame, dict) and isinstance(frame.get('parsed'), dict)]
    caption_vi = _first_text(context, ('caption_vi', 'scene_caption_vi', 'summary_vi', 'description_vi'))
    caption_en = _first_text(context, ('caption_en', 'scene_caption_en', 'summary_en', 'description_en'))
    if not caption_vi:
        caption_vi = normalize_space(' '.join(filter(None, (_first_text(frame, ('detailed_caption_vi', 'short_caption_vi')) for frame in parsed_frames))))
    if not caption_en:
        caption_en = normalize_space(' '.join(filter(None, (_first_text(frame, ('detailed_caption_en', 'short_caption_en')) for frame in parsed_frames))))
    if not caption_vi and (not caption_en):
        caption_vi = normalize_space(item.get('scene_index_text', ''))
    entities = [_flatten_item(frame.get('entities')) for frame in parsed_frames]
    relations = [_flatten_item(frame.get('relations')) for frame in parsed_frames]
    keywords_vi = [_flatten_item(frame.get('keywords_vi')) for frame in parsed_frames]
    keywords_en = [_flatten_item(frame.get('keywords_en')) for frame in parsed_frames]
    visible_text: list[dict[str, str]] = []
    for frame in parsed_frames:
        for region in frame.get('ocr_regions') or []:
            text = _flatten_item(region)
            if text:
                visible_text.append({'text': text})
    parse_ok = item.get('parse_ok')
    status = 'generated' if parse_ok is not False and (caption_vi or caption_en) else 'needs_review'
    return {'scene_id': scene_id, 'status': status, 'quality_errors': _list_text(item.get('error')) if item.get('error') else [], 'semantic': {'caption_vi': caption_vi, 'caption_en': caption_en, 'speech_summary': _first_text(context, ('speech_summary', 'asr_summary')), 'scene_type': _first_text(context, ('scene_type', 'type')) or 'other', 'visible_text': visible_text, 'objects': [text for text in entities if text], 'relations': [text for text in relations if text], 'actions': _list_text(context.get('actions')), 'attributes': _list_text(context.get('attributes')), 'keywords_vi': [text for text in keywords_vi if text], 'keywords_en': [text for text in keywords_en if text], 'temporal_events': context.get('temporal_events') if isinstance(context.get('temporal_events'), list) else []}}

def _load_semantics(roots: list[Path]) -> dict[str, dict]:
    result: dict[str, dict] = {}
    for filename in ('scene_semantics_qwen3vl.jsonl', 'caption_scenes_qwen3vl.jsonl'):
        for path in _discover(roots, filename):
            for item in read_jsonl(path):
                adapted = _adapt_selfhosted_semantic(dict(item))
                if adapted is not None and adapted['scene_id'] not in result:
                    result[adapted['scene_id']] = adapted
    for path in _discover(roots, 'scene_captions_selfhosted.jsonl'):
        for item in read_jsonl(path):
            adapted = _adapt_selfhosted_semantic(dict(item))
            if adapted is not None and adapted['scene_id'] not in result:
                result[adapted['scene_id']] = adapted
    grouped: dict[str, list[dict]] = defaultdict(list)
    for path in _discover(roots, 'scene_text_index_ready_selfhosted.jsonl'):
        for item in read_jsonl(path):
            scene_id = _valid_scene_key(dict(item))
            if scene_id:
                grouped[scene_id].append(dict(item))
    for scene_id, rows in grouped.items():
        if scene_id in result:
            continue
        captions_vi = [_first_text(row, ('detailed_caption_vi', 'short_caption_vi', 'caption_vi')) for row in rows]
        captions_en = [_first_text(row, ('detailed_caption_en', 'short_caption_en', 'caption_en')) for row in rows]
        fallback = normalize_space(' '.join((_flatten_item(row.get('index_text')) for row in rows)))
        result[scene_id] = {'scene_id': scene_id, 'status': 'generated', 'semantic': {'caption_vi': normalize_space(' '.join(filter(None, captions_vi))) or fallback, 'caption_en': normalize_space(' '.join(filter(None, captions_en))), 'objects': [_flatten_item(row.get('entities')) for row in rows], 'relations': [_flatten_item(row.get('relations')) for row in rows], 'keywords_vi': [_flatten_item(row.get('keywords_vi')) for row in rows], 'keywords_en': [_flatten_item(row.get('keywords_en')) for row in rows]}}
    return result

def _iter_scene_validation(value: Any) -> Iterable[dict]:
    if isinstance(value, list):
        for item in value:
            yield from _iter_scene_validation(item)
    elif isinstance(value, dict):
        if value.get('scene_id'):
            yield value
        for key, child in value.items():
            if key not in {'semantic', 'metadata', 'input', 'generation'}:
                yield from _iter_scene_validation(child)

def _load_validations(roots: list[Path]) -> tuple[dict[str, list[dict]], list[dict]]:
    result: dict[str, list[dict]] = defaultdict(list)
    global_reports: list[dict] = []
    candidates = unique_paths((path for root in roots for path in root.rglob('*') if path.is_file() and path.suffix.casefold() in {'.json', '.jsonl'} and ('valid' in path.name.casefold() or 'quality_report' in path.name.casefold()) and ('raw_outputs' not in path.parts)))
    for path in candidates:
        try:
            if path.suffix.casefold() == '.jsonl':
                values: Any = list(read_jsonl(path))
            else:
                values = _read_json(path)
        except Exception:
            continue
        if path.name in {'component_validation_report.json', 'validation_report.json'} and isinstance(values, dict):
            report = dict(values)
            report['_source_file'] = path.name
            global_reports.append(report)
            for kind, status in (('warnings', 'needs_review'), ('errors', 'invalid')):
                for message in _list_text(values.get(kind)):
                    match = SCENE_ID_IN_TEXT.search(message)
                    if match:
                        result[match.group('scene_id')].append({'status': status, kind: [message], 'source': path.name})
        for item in _iter_scene_validation(values):
            result[str(item['scene_id'])].append(item)
    return (dict(result), global_reports)

def _list_text(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        return [normalize_space(value)] if normalize_space(value) else []
    if isinstance(value, dict):
        text = _flatten_item(value)
        return [text] if text else []
    if not isinstance(value, list):
        text = normalize_space(value)
        return [text] if text else []
    output: list[str] = []
    for item in value:
        output.extend(_list_text(item))
    return output

def _semantic_fields(record: dict | None) -> dict[str, Any]:
    record = record or {}
    semantic = record.get('semantic') if isinstance(record.get('semantic'), dict) else record
    subjects: list[str] = []
    subject_actions: list[str] = []
    subject_attributes: list[str] = []
    for subject in semantic.get('subjects', []) or []:
        if not isinstance(subject, dict):
            continue
        description = normalize_space(' '.join(filter(None, [str(subject.get('type', '')), str(subject.get('description', ''))])))
        if description:
            subjects.append(description)
        subject_actions.extend(_list_text(subject.get('action')))
        subject_attributes.extend(_list_text(subject.get('attributes')))
    visible_text = [normalize_space(item.get('text', '') if isinstance(item, dict) else item) for item in semantic.get('visible_text') or []]
    visible_text = [item for item in visible_text if item]
    events: list[dict[str, Any]] = []
    for index, event in enumerate(semantic.get('temporal_events') or [], 1):
        if not isinstance(event, dict):
            continue
        start = max(0.0, float(event.get('start_sec', 0.0) or 0.0))
        end = max(start, float(event.get('end_sec', start) or start))
        events.append({'order': int(event.get('order', index) or index), 'start_sec': start, 'end_sec': end, 'description_vi': normalize_space(event.get('description_vi', '')), 'description_en': normalize_space(event.get('description_en', ''))})
    objects = _list_text(semantic.get('objects'))
    actions = _list_text(semantic.get('actions')) + subject_actions
    attributes = _list_text(semantic.get('attributes')) + subject_attributes
    keywords = _list_text(semantic.get('keywords_vi')) + _list_text(semantic.get('keywords_en'))
    return {'caption_vi': normalize_space(semantic.get('caption_vi', '')), 'caption_en': normalize_space(semantic.get('caption_en', '')), 'speech_summary': normalize_space(semantic.get('speech_summary', '')), 'scene_type': normalize_space(semantic.get('scene_type', 'other')) or 'other', 'visible_text': normalize_space(' '.join(visible_text)), 'keywords': normalize_space(' '.join(keywords)), 'entities': normalize_space(' '.join([*subjects, *objects])), 'actions': normalize_space(' '.join(actions)), 'attributes': normalize_space(' '.join(attributes)), 'relations': normalize_space(' '.join(_list_text(semantic.get('relations')))), 'event_text': normalize_space(' '.join((text for event in events for text in (event['description_vi'], event['description_en']) if text))), 'temporal_events': events, 'semantic_status': str(record.get('status', 'generated' if semantic else 'missing')), 'semantic_errors': _list_text(record.get('quality_errors'))}

def _quality_state(semantic: dict | None, validations: list[dict], needs_review_penalty: float) -> tuple[str, float, list[str]]:
    statuses: list[str] = []
    errors: list[str] = []
    if semantic:
        statuses.append(str(semantic.get('status', '')))
        errors.extend(_list_text(semantic.get('quality_errors')))
    for item in validations:
        for key in ('status', 'validation_status', 'quality_status', 'result'):
            if key in item:
                statuses.append(str(item[key]))
        if item.get('valid') is False or item.get('passed') is False:
            statuses.append('invalid')
        errors.extend(_list_text(item.get('errors')))
        errors.extend(_list_text(item.get('quality_errors')))
        errors.extend(_list_text(item.get('warnings')))
    normalized = ' '.join((status.casefold() for status in statuses))
    if any((token in normalized for token in ('invalid', 'failed', 'error'))):
        return ('invalid', 0.0, list(dict.fromkeys(errors)))
    if 'needs_review' in normalized or 'warning' in normalized or errors:
        return ('needs_review', needs_review_penalty, list(dict.fromkeys(errors)))
    return ('passed', 1.0, [])

def load_components(input_root: Path, staging_dir: Path, needs_review_penalty: float=0.75) -> LoadedComponents:
    roots = _prepare_roots(input_root, staging_dir)
    warnings: list[str] = []
    keyframes, keyframe_embeddings, keyframe_model = _load_keyframes(roots)
    ocr_frames = _load_scene_text(roots, 'ocr_keyframes.jsonl', key_field='keyframe_id')
    for frame in keyframes:
        frame.ocr_text = normalize_space(ocr_frames.get(frame.keyframe_id, {}).get('text', ''))
    ocr_scenes = _load_scene_text(roots, 'ocr_scenes.jsonl')
    asr_scenes = _load_scene_text(roots, 'asr_scenes.jsonl')
    asr_segments = _load_asr_segments(roots)
    manifests = _load_scene_manifests(roots)
    metadata, metadata_vectors, scene_model = _load_metadata(roots)
    semantics = _load_semantics(roots)
    validations, validation_reports = _load_validations(roots)
    failed_reports = [report for report in validation_reports if report.get('passed') is False or str(report.get('status', '')).casefold() in {'failed', 'invalid', 'error'}]
    if failed_reports:
        errors = [message for report in failed_reports for message in _list_text(report.get('errors'))]
        detail = '; '.join(errors[:8]) or 'component compatibility check failed'
        raise ValueError(f'Output 06 validation failed: {detail}')
    frames_by_scene: dict[str, list[KeyframeDocument]] = defaultdict(list)
    for frame in keyframes:
        frames_by_scene[frame.scene_id].append(frame)
    if not metadata:
        warnings.append('scene_metadata.jsonl was not found; scene metadata was rebuilt from outputs 01-04.')
        if not manifests:
            raise FileNotFoundError('Need scene_metadata.jsonl or scene_clip_manifest.json')
        if keyframe_embeddings is None:
            raise FileNotFoundError('Need keyframe embeddings to rebuild scene embeddings')
        for scene_id, manifest in manifests.items():
            video_id, scene_no = scene_parts(scene_id)
            frames = sorted(frames_by_scene.get(scene_id, []), key=lambda item: item.frame_idx)
            if not frames:
                raise ValueError(f'{scene_id}: no keyframes')
            local_vectors = np.stack([keyframe_embeddings[frame.vector_row] for frame in frames])
            vector = local_vectors.mean(axis=0)
            vector /= np.linalg.norm(vector)
            metadata_vectors[scene_id] = vector
            representative = max(frames, key=lambda item: item.quality_score)
            start_sec = float(manifest.get('start_sec_absolute', frames[0].timestamp_sec))
            end_sec = float(manifest.get('end_sec_absolute', frames[-1].timestamp_sec))
            metadata[scene_id] = {'scene_id': scene_id, 'video_id': video_id, 'scene_index': scene_no, 'start_frame': frames[0].frame_idx, 'end_frame': frames[-1].frame_idx, 'start_sec': start_sec, 'end_sec': end_sec, 'clip_path': str(manifest.get('clip_path', '')), 'representative_keyframe_id': representative.keyframe_id, 'ocr_text': normalize_space(ocr_scenes.get(scene_id, {}).get('text', '')), 'transcript_text': normalize_space(asr_scenes.get(scene_id, {}).get('text', ''))}
        scene_model = keyframe_model
    aligned_asr = _align_asr_segments(metadata, asr_segments) if asr_segments else {}
    existing_keyframe_ids = {frame.keyframe_id for frame in keyframes}
    for scene_id, item in metadata.items():
        for raw in item.get('keyframes', []) or []:
            keyframe_id = str(raw.get('keyframe_id', ''))
            if not keyframe_id or keyframe_id in existing_keyframe_ids:
                continue
            frame = KeyframeDocument(keyframe_id=keyframe_id, scene_id=scene_id, frame_idx=int(raw.get('frame_idx', 0)), timestamp_sec=float(raw.get('timestamp_sec', item.get('start_sec', 0.0))), image_path=str(raw.get('image_path', '')), vector_row=len(keyframes), metadata={'source': 'scene_metadata.jsonl'})
            keyframes.append(frame)
            frames_by_scene[scene_id].append(frame)
            existing_keyframe_ids.add(keyframe_id)
    ordered_ids = sorted(metadata, key=scene_parts)
    scene_vectors: list[np.ndarray] = []
    scenes: list[SceneDocument] = []
    for scene_id in ordered_ids:
        item = metadata[scene_id]
        vector = np.asarray(metadata_vectors[scene_id], dtype=np.float32)
        norm = float(np.linalg.norm(vector))
        if not np.isfinite(norm) or norm < 1e-08:
            raise ValueError(f'{scene_id}: invalid scene embedding')
        scene_vectors.append(vector / norm)
        video_id, scene_no = scene_parts(scene_id)
        semantic = semantics.get(scene_id)
        fields = _semantic_fields(semantic)
        quality_status, quality_penalty, quality_errors = _quality_state(semantic, validations.get(scene_id, []), needs_review_penalty)
        manifest = manifests.get(scene_id, {})
        clip_path = str(item.get('clip_path') or manifest.get('clip_path') or '')
        scenes.append(SceneDocument(scene_id=scene_id, video_id=str(item.get('video_id', video_id)), scene_no=int(item.get('scene_index', scene_no)), start_sec=float(item['start_sec']), end_sec=float(item['end_sec']), clip_path=clip_path, start_frame=int(item.get('start_frame', 0)), end_frame=int(item.get('end_frame', 0)), representative_keyframe_id=str(item.get('representative_keyframe_id', '')), vector_row=len(scene_vectors) - 1, ocr_text=normalize_space(item.get('ocr_text') or ocr_scenes.get(scene_id, {}).get('text', '')), transcript=normalize_space(aligned_asr.get(scene_id, '') if asr_segments else item.get('transcript_text') or asr_scenes.get(scene_id, {}).get('text', '')), **{key: value for key, value in fields.items() if key != 'semantic_errors'}, quality_status=quality_status, quality_penalty=quality_penalty, quality_errors=quality_errors, metadata={'component_status': item.get('component_status', {}), 'provenance': item.get('provenance', {}), 'semantic_generation': (semantic or {}).get('generation', {}), 'semantic_uncertainty': ((semantic or {}).get('semantic') or {}).get('uncertainty', []) if isinstance((semantic or {}).get('semantic'), dict) else [], 'validation': validations.get(scene_id, [])}))
    matrix = np.ascontiguousarray(np.stack(scene_vectors), dtype=np.float32)
    if semantics and set(metadata) - set(semantics):
        warnings.append(f'Missing semantics for {len(set(metadata) - set(semantics))} scenes.')
    if not semantics:
        warnings.append('Output 05 semantics was not found; semantic/tag/event branches are sparse.')
    component_reports = [report for report in validation_reports if report.get('_source_file') == 'component_validation_report.json']
    if not component_reports:
        warnings.append('Output 06 component_validation_report.json was not found.')
    stats = {'source_roots': [str(path) for path in roots], 'scene_count': len(scenes), 'keyframe_count': len(keyframes), 'video_count': len({scene.video_id for scene in scenes}), 'semantic_scene_count': len(semantics), 'validation_scene_count': len(validations), 'validation_report_count': len(validation_reports), 'validation_passed': bool(component_reports) and all((report.get('passed') is True for report in component_reports)), 'passed_scene_count': sum((scene.quality_status == 'passed' for scene in scenes)), 'needs_review_scene_count': sum((scene.quality_status == 'needs_review' for scene in scenes)), 'invalid_scene_count': sum((scene.quality_status == 'invalid' for scene in scenes))}
    return LoadedComponents(scenes=scenes, keyframes=keyframes, scene_embeddings=matrix, keyframe_embeddings=keyframe_embeddings, scene_embedding_model=scene_model, keyframe_embedding_model=keyframe_model, embedding_dimension=int(matrix.shape[1]), source_root=input_root.resolve(), stats=stats, warnings=warnings)

## 5. SQLite FTS5 và metadata store

Năm nhánh lexical được lưu riêng: `semantic`, `ocr`, `speech`, `tags`,
`event`. Candidate phải vượt lọc độ phủ từ khóa trước khi được đưa vào RRF.

In [78]:
from __future__ import annotations

# ===========================================================================
# Logic preserved from: storage.py
# ===========================================================================
import json
import sqlite3
from pathlib import Path
from typing import Iterable
SCHEMA_SQL = "\nPRAGMA foreign_keys = ON;\n\nCREATE TABLE scenes (\n    scene_id TEXT PRIMARY KEY,\n    video_id TEXT NOT NULL,\n    scene_no INTEGER NOT NULL,\n    start_frame INTEGER NOT NULL,\n    end_frame INTEGER NOT NULL,\n    start_sec REAL NOT NULL,\n    end_sec REAL NOT NULL,\n    clip_path TEXT NOT NULL DEFAULT '',\n    representative_keyframe_id TEXT NOT NULL DEFAULT '',\n    vector_row INTEGER NOT NULL UNIQUE,\n    ocr_text TEXT NOT NULL DEFAULT '',\n    transcript TEXT NOT NULL DEFAULT '',\n    caption_vi TEXT NOT NULL DEFAULT '',\n    caption_en TEXT NOT NULL DEFAULT '',\n    speech_summary TEXT NOT NULL DEFAULT '',\n    scene_type TEXT NOT NULL DEFAULT 'other',\n    visible_text TEXT NOT NULL DEFAULT '',\n    keywords TEXT NOT NULL DEFAULT '',\n    entities TEXT NOT NULL DEFAULT '',\n    actions TEXT NOT NULL DEFAULT '',\n    attributes TEXT NOT NULL DEFAULT '',\n    relations TEXT NOT NULL DEFAULT '',\n    event_text TEXT NOT NULL DEFAULT '',\n    semantic_status TEXT NOT NULL DEFAULT 'missing',\n    quality_status TEXT NOT NULL DEFAULT 'passed',\n    quality_penalty REAL NOT NULL DEFAULT 1.0,\n    quality_errors_json TEXT NOT NULL DEFAULT '[]',\n    metadata_json TEXT NOT NULL DEFAULT '{}'\n);\n\nCREATE INDEX idx_scenes_video_time ON scenes(video_id, start_sec, end_sec);\nCREATE INDEX idx_scenes_quality ON scenes(quality_status);\n\nCREATE TABLE keyframes (\n    keyframe_id TEXT PRIMARY KEY,\n    scene_id TEXT NOT NULL REFERENCES scenes(scene_id),\n    frame_idx INTEGER NOT NULL,\n    timestamp_sec REAL NOT NULL,\n    image_path TEXT NOT NULL,\n    vector_row INTEGER NOT NULL UNIQUE,\n    quality_score REAL NOT NULL DEFAULT 0,\n    ocr_text TEXT NOT NULL DEFAULT '',\n    metadata_json TEXT NOT NULL DEFAULT '{}'\n);\nCREATE INDEX idx_keyframes_scene_time ON keyframes(scene_id, timestamp_sec);\n\nCREATE TABLE events (\n    event_id TEXT PRIMARY KEY,\n    scene_id TEXT NOT NULL REFERENCES scenes(scene_id),\n    video_id TEXT NOT NULL,\n    event_order INTEGER NOT NULL,\n    relative_start_sec REAL NOT NULL,\n    relative_end_sec REAL NOT NULL,\n    absolute_start_sec REAL NOT NULL,\n    absolute_end_sec REAL NOT NULL,\n    description_vi TEXT NOT NULL DEFAULT '',\n    description_en TEXT NOT NULL DEFAULT ''\n);\nCREATE INDEX idx_events_video_time ON events(video_id, absolute_start_sec, absolute_end_sec);\n\nCREATE TABLE engine_meta (key TEXT PRIMARY KEY, value_json TEXT NOT NULL);\n\nCREATE VIRTUAL TABLE semantic_fts USING fts5(\n    scene_id UNINDEXED, caption_vi, caption_en, event_text,\n    tokenize='unicode61 remove_diacritics 2', prefix='2 3 4'\n);\nCREATE VIRTUAL TABLE ocr_fts USING fts5(\n    scene_id UNINDEXED, ocr_text, visible_text,\n    tokenize='unicode61 remove_diacritics 2', prefix='2 3 4'\n);\nCREATE VIRTUAL TABLE speech_fts USING fts5(\n    scene_id UNINDEXED, transcript, speech_summary,\n    tokenize='unicode61 remove_diacritics 2', prefix='2 3 4'\n);\nCREATE VIRTUAL TABLE tags_fts USING fts5(\n    scene_id UNINDEXED, keywords, entities, actions, attributes, relations, scene_type,\n    tokenize='unicode61 remove_diacritics 2', prefix='2 3 4'\n);\nCREATE VIRTUAL TABLE event_fts USING fts5(\n    event_id UNINDEXED, scene_id UNINDEXED, description_vi, description_en,\n    tokenize='unicode61 remove_diacritics 2', prefix='2 3 4'\n);\n"
BRANCHES = {'semantic': ('semantic_fts', (0.0, 2.0, 1.5, 1.2)), 'ocr': ('ocr_fts', (0.0, 2.0, 1.2)), 'speech': ('speech_fts', (0.0, 2.0, 1.4)), 'tags': ('tags_fts', (0.0, 1.5, 1.4, 1.4, 1.0, 1.0, 0.8))}
BRANCH_EVIDENCE_FIELDS = {'semantic': ('caption_vi', 'caption_en', 'event_text'), 'ocr': ('ocr_text', 'visible_text'), 'speech': ('transcript', 'speech_summary'), 'tags': ('keywords', 'entities', 'actions', 'attributes', 'relations', 'scene_type')}

def _fts_text(value: str) -> str:
    """Store original and accent-folded text (notably Vietnamese ``đ``)."""
    folded = accent_fold(value)
    return value if folded == value else f'{value} {folded}'

def connect_database(path: Path, readonly: bool=False) -> sqlite3.Connection:
    connection = sqlite3.connect(f'file:{path.resolve()}?mode=ro', uri=True) if readonly else sqlite3.connect(path)
    connection.row_factory = sqlite3.Row
    connection.execute('PRAGMA foreign_keys = ON')
    connection.execute('PRAGMA temp_store = MEMORY')
    connection.execute('PRAGMA cache_size = -65536')
    return connection

def create_database(path: Path, scenes: Iterable[SceneDocument], keyframes: Iterable[KeyframeDocument], meta: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    connection = connect_database(path)
    try:
        connection.executescript(SCHEMA_SQL)
        scene_list = list(scenes)
        connection.executemany(f"INSERT INTO scenes VALUES ({','.join(('?' for _ in range(28)))})", [(scene.scene_id, scene.video_id, scene.scene_no, scene.start_frame, scene.end_frame, scene.start_sec, scene.end_sec, scene.clip_path, scene.representative_keyframe_id, scene.vector_row, scene.ocr_text, scene.transcript, scene.caption_vi, scene.caption_en, scene.speech_summary, scene.scene_type, scene.visible_text, scene.keywords, scene.entities, scene.actions, scene.attributes, scene.relations, scene.event_text, scene.semantic_status, scene.quality_status, scene.quality_penalty, json.dumps(scene.quality_errors, ensure_ascii=False), json.dumps(scene.metadata, ensure_ascii=False)) for scene in scene_list])
        connection.executemany('INSERT INTO semantic_fts VALUES (?,?,?,?)', [(s.scene_id, _fts_text(s.caption_vi), _fts_text(s.caption_en), _fts_text(s.event_text)) for s in scene_list])
        connection.executemany('INSERT INTO ocr_fts VALUES (?,?,?)', [(s.scene_id, _fts_text(s.ocr_text), _fts_text(s.visible_text)) for s in scene_list])
        connection.executemany('INSERT INTO speech_fts VALUES (?,?,?)', [(s.scene_id, _fts_text(s.transcript), _fts_text(s.speech_summary)) for s in scene_list])
        connection.executemany('INSERT INTO tags_fts VALUES (?,?,?,?,?,?,?)', [(s.scene_id, _fts_text(s.keywords), _fts_text(s.entities), _fts_text(s.actions), _fts_text(s.attributes), _fts_text(s.relations), _fts_text(s.scene_type)) for s in scene_list])
        event_rows = []
        event_fts_rows = []
        for scene in scene_list:
            for index, event in enumerate(scene.temporal_events, 1):
                order = int(event.get('order', index))
                event_id = f'{scene.scene_id}_E{order:04d}'
                rel_start = float(event.get('start_sec', 0.0))
                rel_end = float(event.get('end_sec', rel_start))
                desc_vi = str(event.get('description_vi', ''))
                desc_en = str(event.get('description_en', ''))
                event_rows.append((event_id, scene.scene_id, scene.video_id, order, rel_start, rel_end, scene.start_sec + rel_start, scene.start_sec + rel_end, desc_vi, desc_en))
                event_fts_rows.append((event_id, scene.scene_id, _fts_text(desc_vi), _fts_text(desc_en)))
        connection.executemany('INSERT INTO events VALUES (?,?,?,?,?,?,?,?,?,?)', event_rows)
        connection.executemany('INSERT INTO event_fts VALUES (?,?,?,?)', event_fts_rows)
        scene_ids = {scene.scene_id for scene in scene_list}
        frame_rows = []
        for frame in keyframes:
            if frame.scene_id not in scene_ids:
                continue
            frame_rows.append((frame.keyframe_id, frame.scene_id, frame.frame_idx, frame.timestamp_sec, frame.image_path, frame.vector_row, frame.quality_score, frame.ocr_text, json.dumps(frame.metadata, ensure_ascii=False)))
        connection.executemany('INSERT INTO keyframes VALUES (?,?,?,?,?,?,?,?,?)', frame_rows)
        connection.executemany('INSERT INTO engine_meta(key,value_json) VALUES (?,?)', [(key, json.dumps(value, ensure_ascii=False)) for key, value in meta.items()])
        for table in (*[value[0] for value in BRANCHES.values()], 'event_fts'):
            connection.execute(f"INSERT INTO {table}({table}) VALUES ('optimize')")
        connection.commit()
    finally:
        connection.close()

def _filter_sql(video_id: str | None, start_sec: float | None, end_sec: float | None, exclude_invalid: bool) -> tuple[list[str], list[object]]:
    where: list[str] = []
    params: list[object] = []
    if video_id:
        where.append('s.video_id = ?')
        params.append(video_id)
    if start_sec is not None:
        where.append('s.end_sec >= ?')
        params.append(float(start_sec))
    if end_sec is not None:
        where.append('s.start_sec <= ?')
        params.append(float(end_sec))
    if exclude_invalid:
        where.append("s.quality_status != 'invalid'")
    return (where, params)

def search_branch(connection: sqlite3.Connection, branch: str, text: str, limit: int, config: EngineConfig, video_id: str | None=None, start_sec: float | None=None, end_sec: float | None=None, match_all: bool=False) -> list[dict]:
    if branch not in BRANCHES:
        raise ValueError(f'Unknown lexical branch: {branch}')
    query = make_fts_query(text, match_all=match_all)
    if not query:
        return []
    table, weights = BRANCHES[branch]
    where, params = _filter_sql(video_id, start_sec, end_sec, config.exclude_invalid)
    where.insert(0, f'{table} MATCH ?')
    params.insert(0, query)
    placeholders = ','.join(('?' for _ in weights))
    sql = f"\n        SELECT s.*, bm25({table},{placeholders}) AS distance,\n               snippet({table},-1,'[',']',' … ',28) AS snippet\n        FROM {table} JOIN scenes s ON s.scene_id={table}.scene_id\n        WHERE {' AND '.join(where)}\n        ORDER BY distance ASC LIMIT ?\n    "
    rows = connection.execute(sql, [*weights, *params, int(limit)]).fetchall()
    terms = query_terms(text)
    required_terms = min(config.min_lexical_terms, len(terms))
    required_coverage = 1.0 if match_all else config.min_lexical_coverage
    output: list[dict] = []
    for row in rows:
        evidence = ' '.join((str(row[field] or '') for field in BRANCH_EVIDENCE_FIELDS[branch]))
        coverage, matched_terms = lexical_coverage(text, evidence)
        if len(matched_terms) < required_terms or coverage + 1e-12 < required_coverage:
            continue
        output.append({'scene_id': row['scene_id'], 'branch': branch, 'score': -float(row['distance']), 'snippet': row['snippet'] or '', 'lexical_coverage': float(coverage), 'matched_terms': matched_terms})
    return output

def search_event_branch(connection: sqlite3.Connection, text: str, limit: int, config: EngineConfig, video_id: str | None=None, start_sec: float | None=None, end_sec: float | None=None, match_all: bool=False) -> list[dict]:
    query = make_fts_query(text, match_all=match_all)
    if not query:
        return []
    where, params = _filter_sql(video_id, start_sec, end_sec, config.exclude_invalid)
    where.insert(0, 'event_fts MATCH ?')
    params.insert(0, query)
    rows = connection.execute(f"SELECT e.*, bm25(event_fts,0.0,0.0,2.0,1.5) AS distance,\n                   snippet(event_fts,-1,'[',']',' … ',24) AS snippet\n            FROM event_fts\n            JOIN events e ON e.event_id=event_fts.event_id\n            JOIN scenes s ON s.scene_id=e.scene_id\n            WHERE {' AND '.join(where)}\n            ORDER BY distance ASC LIMIT ?", [*params, int(limit * 3)]).fetchall()
    output: list[dict] = []
    seen: set[str] = set()
    terms = query_terms(text)
    required_terms = min(config.min_lexical_terms, len(terms))
    required_coverage = 1.0 if match_all else config.min_lexical_coverage
    for row in rows:
        if row['scene_id'] in seen:
            continue
        evidence = ' '.join(filter(None, [str(row['description_vi'] or ''), str(row['description_en'] or '')]))
        coverage, matched_terms = lexical_coverage(text, evidence)
        if len(matched_terms) < required_terms or coverage + 1e-12 < required_coverage:
            continue
        seen.add(row['scene_id'])
        output.append({'scene_id': row['scene_id'], 'branch': 'event', 'score': -float(row['distance']), 'snippet': row['snippet'] or '', 'lexical_coverage': float(coverage), 'matched_terms': matched_terms, 'matched_event': {key: row[key] for key in ('event_id', 'event_order', 'relative_start_sec', 'relative_end_sec', 'absolute_start_sec', 'absolute_end_sec', 'description_vi', 'description_en')}})
        if len(output) >= limit:
            break
    return output

def _decode_scene(row: sqlite3.Row) -> dict:
    item = dict(row)
    item['quality_errors'] = json.loads(item.pop('quality_errors_json'))
    item['metadata'] = json.loads(item.pop('metadata_json'))
    return item

def fetch_scenes(connection: sqlite3.Connection, scene_ids: list[str]) -> dict[str, dict]:
    if not scene_ids:
        return {}
    placeholders = ','.join(('?' for _ in scene_ids))
    rows = connection.execute(f'SELECT * FROM scenes WHERE scene_id IN ({placeholders})', scene_ids).fetchall()
    return {row['scene_id']: _decode_scene(row) for row in rows}

def fetch_scenes_by_vector_rows(connection: sqlite3.Connection, vector_rows: list[int]) -> dict[int, dict]:
    if not vector_rows:
        return {}
    placeholders = ','.join(('?' for _ in vector_rows))
    rows = connection.execute(f'SELECT * FROM scenes WHERE vector_row IN ({placeholders})', vector_rows).fetchall()
    return {int(row['vector_row']): _decode_scene(row) for row in rows}

def fetch_frames_by_vector_rows(connection: sqlite3.Connection, vector_rows: list[int]) -> dict[int, dict]:
    if not vector_rows:
        return {}
    placeholders = ','.join(('?' for _ in vector_rows))
    rows = connection.execute(f'SELECT * FROM keyframes WHERE vector_row IN ({placeholders})', vector_rows).fetchall()
    output = {}
    for row in rows:
        item = dict(row)
        item['metadata'] = json.loads(item.pop('metadata_json'))
        output[int(item['vector_row'])] = item
    return output

def representative_frame(connection: sqlite3.Connection, scene_id: str) -> dict | None:
    row = connection.execute('SELECT k.* FROM keyframes k JOIN scenes s ON s.scene_id=k.scene_id\n           WHERE k.scene_id=?\n           ORDER BY (k.keyframe_id=s.representative_keyframe_id) DESC,\n                    k.quality_score DESC, ABS(k.timestamp_sec-(s.start_sec+s.end_sec)/2.0) ASC\n           LIMIT 1', (scene_id,)).fetchone()
    if row is None:
        return None
    item = dict(row)
    item['metadata'] = json.loads(item.pop('metadata_json'))
    return item

## 6. Vector index

`VectorIndex` đọc FAISS HNSW khi có; nếu không sẽ dùng ma trận NumPy đã chuẩn
hóa. `OpenClipTextEncoder` được khởi tạo lười, chỉ khi thật sự có visual query.

In [79]:
from __future__ import annotations

# ===========================================================================
# Logic preserved from: vector_index.py
# ===========================================================================
import json
from pathlib import Path
from typing import Any
import numpy as np

def _try_import_faiss() -> Any | None:
    try:
        import faiss
        return faiss
    except ImportError:
        return None

def normalize_vectors(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=np.float32)
    if matrix.ndim == 1:
        matrix = matrix[None, :]
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if np.any(~np.isfinite(norms)) or np.any(norms < 1e-08):
        raise ValueError('Vector contains NaN, infinity, or has zero norm')
    return np.ascontiguousarray(matrix / norms, dtype=np.float32)

def build_vector_index(index_dir: Path, embeddings: np.ndarray, config: EngineConfig, name: str) -> dict:
    config.validate()
    index_dir.mkdir(parents=True, exist_ok=True)
    embeddings = normalize_vectors(embeddings)
    numpy_path = index_dir / f'{name}_embeddings.npy'
    faiss = _try_import_faiss()
    requested = config.vector_backend
    backend = 'numpy'
    faiss_path = index_dir / f'{name}_hnsw.faiss'
    if requested == 'faiss' and faiss is None:
        raise RuntimeError("vector_backend='faiss' was requested but faiss is not installed. Install faiss-cpu with Conda or use vector_backend='numpy'.")
    if faiss is not None and requested in {'auto', 'faiss'}:
        dimension = int(embeddings.shape[1])
        index = faiss.IndexHNSWFlat(dimension, config.hnsw_m, faiss.METRIC_INNER_PRODUCT)
        index.hnsw.efConstruction = config.hnsw_ef_construction
        index.hnsw.efSearch = config.hnsw_ef_search
        index.add(embeddings)
        faiss.write_index(index, str(faiss_path))
        backend = 'faiss_hnsw'
    elif faiss_path.exists():
        faiss_path.unlink()
    save_numpy = backend == 'numpy' or config.keep_numpy_fallback
    if save_numpy:
        np.save(numpy_path, embeddings)
    elif numpy_path.exists():
        numpy_path.unlink()
    return {'name': name, 'backend': backend, 'count': int(embeddings.shape[0]), 'dimension': int(embeddings.shape[1]), 'metric': 'cosine_via_normalized_inner_product', 'numpy_file': numpy_path.name if save_numpy else None, 'faiss_file': faiss_path.name if backend == 'faiss_hnsw' else None, 'hnsw': {'m': config.hnsw_m, 'ef_construction': config.hnsw_ef_construction, 'ef_search': config.hnsw_ef_search}}

class VectorIndex:

    def __init__(self, index_dir: Path, manifest: dict):
        self.index_dir = index_dir
        self.manifest = manifest
        self.dimension = int(manifest['dimension'])
        self.backend = str(manifest['backend'])
        self._faiss = None
        self._index = None
        self._matrix = None
        if self.backend == 'faiss_hnsw':
            self._faiss = _try_import_faiss()
            if self._faiss is not None:
                self._index = self._faiss.read_index(str(index_dir / manifest['faiss_file']))
        if self._index is None:
            self.backend = 'numpy'
            if not manifest.get('numpy_file'):
                raise RuntimeError(f"FAISS index {manifest.get('faiss_file')} exists but faiss is not installed, and no Numpy fallback was retained.")
            self._matrix = np.load(index_dir / manifest['numpy_file'], mmap_mode='r')

    @classmethod
    def from_directory(cls, index_dir: Path, name: str='scene') -> 'VectorIndex':
        manifest = json.loads((index_dir / 'index_manifest.json').read_text(encoding='utf-8'))
        return cls(index_dir, manifest[f'{name}_vector_index'])

    def search(self, query_vector: np.ndarray, k: int) -> list[tuple[int, float]]:
        query = normalize_vectors(np.asarray(query_vector, dtype=np.float32))
        if query.shape[1] != self.dimension:
            raise ValueError(f'Query vector dimension {query.shape[1]} != index dimension {self.dimension}')
        k = max(0, min(int(k), int(self.manifest['count'])))
        if k == 0:
            return []
        if self._index is not None:
            scores, indices = self._index.search(query, k)
            return [(int(index), float(score)) for index, score in zip(indices[0], scores[0]) if int(index) >= 0]
        assert self._matrix is not None
        scores = np.asarray(self._matrix @ query[0], dtype=np.float32)
        if k == len(scores):
            indices = np.argsort(-scores)
        else:
            candidates = np.argpartition(-scores, k - 1)[:k]
            indices = candidates[np.argsort(-scores[candidates])]
        return [(int(index), float(scores[index])) for index in indices]

class OpenClipTextEncoder:
    """Lazy query encoder matching ``open_clip:ViT-B-32:openai`` embeddings."""

    def __init__(self, model_spec: str, device: str | None=None):
        try:
            import open_clip
            import torch
        except ImportError as exc:
            raise RuntimeError('Visual text search needs PyTorch and open_clip_torch. Install a CUDA-enabled PyTorch build, then `pip install open_clip_torch`.') from exc
        parts = model_spec.split(':')
        if len(parts) != 3 or parts[0] != 'open_clip':
            raise ValueError(f'Unsupported visual embedding model: {model_spec}')
        _, architecture, pretrained = parts
        self.torch = torch
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model, _, _ = open_clip.create_model_and_transforms(architecture, pretrained=pretrained, device=self.device)
        self.model.eval()
        self.tokenizer = open_clip.get_tokenizer(architecture)

    def encode(self, text: str) -> np.ndarray:
        tokens = self.tokenizer([text]).to(self.device)
        with self.torch.inference_mode():
            vector = self.model.encode_text(tokens)
            vector = vector / vector.norm(dim=-1, keepdim=True)
        return vector.detach().cpu().numpy()[0].astype(np.float32)

## 7. Query planner, fusion và build index

Planner tăng/giảm trọng số theo loại query. Weighted RRF chỉ gộp thứ hạng của
các nhánh sau khi candidate đã qua filter; quality penalty được áp dụng theo
Output 06.

In [80]:
from __future__ import annotations

# ===========================================================================
# Logic preserved from: planner.py
# ===========================================================================
import re
from dataclasses import dataclass

@dataclass(slots=True)
class QueryPlan:
    task: str
    branch_weights: dict[str, float]
    hints: list[str]
OCR_HINTS = ('chữ', 'văn bản', 'biển', 'logo', 'tiêu đề', 'phụ đề', 'màn hình', 'số điện thoại', 'đọc được', 'written', 'text', 'sign', 'subtitle')
SPEECH_HINTS = ('nói', 'phát biểu', 'trả lời', 'hỏi', 'âm thanh', 'giọng', 'nghe', 'said', 'says', 'speech', 'announces', 'mentions')
TEMPORAL_HINTS = ('sau đó', 'trước khi', 'tiếp theo', 'đầu tiên', 'cuối cùng', ' rồi ', 'before', 'after', 'then', 'next', 'finally')
ACTION_HINTS = ('đang', 'hành động', 'di chuyển', 'đi', 'chạy', 'cầm', 'mở', 'đóng', 'action', 'moving', 'walking', 'running', 'holding')

def _contains(text: str, hints: tuple[str, ...]) -> bool:
    padded = f' {text.casefold()} '
    return any((hint in padded for hint in hints))

def plan_query(text: str, config: EngineConfig, task: str='auto') -> QueryPlan:
    task = task.casefold().strip()
    if task not in {'auto', 'frame', 'scene', 'temporal', 'qa'}:
        raise ValueError(f'Unsupported task: {task}')
    weights = {'semantic': config.semantic_weight, 'ocr': config.ocr_weight, 'speech': config.speech_weight, 'tags': config.tags_weight, 'event': config.event_weight, 'scene_vector': config.scene_vector_weight, 'frame_vector': config.frame_vector_weight}
    hints: list[str] = []
    if _contains(text, OCR_HINTS) or bool(re.search('\\b\\d{2,}\\b', text)):
        weights['ocr'] *= 1.8
        hints.append('ocr')
    if _contains(text, SPEECH_HINTS):
        weights['speech'] *= 1.8
        hints.append('speech')
    if task == 'temporal' or _contains(text, TEMPORAL_HINTS):
        weights['event'] *= 1.8
        weights['tags'] *= 1.25
        hints.append('temporal')
    if _contains(text, ACTION_HINTS):
        weights['tags'] *= 1.45
        weights['event'] *= 1.25
        hints.append('action')
    if task == 'frame':
        weights['frame_vector'] *= 1.5
        weights['scene_vector'] *= 0.8
        hints.append('frame')
    elif task in {'scene', 'qa'}:
        weights['semantic'] *= 1.2
        weights['scene_vector'] *= 1.2
    return QueryPlan(task=task, branch_weights=weights, hints=hints)

def split_temporal_query(text: str) -> list[str]:
    parts = re.split('\\s+(?:sau đó|tiếp theo|rồi|trước khi|then|next|after that|before)\\s+', text, flags=re.IGNORECASE)
    return [part.strip(' ,.;') for part in parts if part.strip(' ,.;')]

# ===========================================================================
# Logic preserved from: fusion.py
# ===========================================================================
from collections import defaultdict
from collections.abc import Sequence

def reciprocal_rank_fusion(ranked_lists: Sequence[tuple[Sequence[str], float]], rrf_k: int=60, item_multipliers: dict[str, float] | None=None) -> list[tuple[str, float]]:
    """Fuse ranked ids without trying to calibrate heterogeneous raw scores."""
    if rrf_k < 0:
        raise ValueError('rrf_k must be non-negative')
    scores: dict[str, float] = defaultdict(float)
    best_rank: dict[str, int] = {}
    item_multipliers = item_multipliers or {}
    for ids, weight in ranked_lists:
        for rank, item_id in enumerate(ids, start=1):
            multiplier = float(item_multipliers.get(item_id, 1.0))
            scores[item_id] += float(weight) * multiplier / (rrf_k + rank)
            best_rank[item_id] = min(rank, best_rank.get(item_id, rank))
    return sorted(scores.items(), key=lambda item: (-item[1], best_rank[item[0]], item[0]))

# ===========================================================================
# Logic preserved from: builder.py
# ===========================================================================
import json
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np

def build_index(input_root: str | Path, index_dir: str | Path, config: EngineConfig | None=None) -> BuildReport:
    """Build a complete local lexical/vector/metadata index."""
    started = time.perf_counter()
    config = config or EngineConfig()
    config.validate()
    input_root = Path(input_root).expanduser().resolve()
    index_dir = Path(index_dir).expanduser().resolve()
    index_dir.mkdir(parents=True, exist_ok=True)
    staging_dir = index_dir.parent / '_aic_search_input_cache'
    loaded = load_components(input_root, staging_dir, needs_review_penalty=config.needs_review_penalty)
    scene_vector_manifest = build_vector_index(index_dir, loaded.scene_embeddings, config, name='scene')
    frame_vector_manifest = None
    if loaded.keyframe_embeddings is not None and loaded.keyframes:
        frame_vector_manifest = build_vector_index(index_dir, loaded.keyframe_embeddings, config, name='frame')
    meta = {'schema_version': 2, 'built_at_utc': datetime.now(timezone.utc).isoformat(), 'scene_embedding_model': loaded.scene_embedding_model, 'keyframe_embedding_model': loaded.keyframe_embedding_model, 'embedding_dimension': loaded.embedding_dimension, 'source_root': str(loaded.source_root), 'stats': loaded.stats, 'warnings': loaded.warnings, 'config': config.to_dict()}
    database_path = index_dir / 'aic_search.db'
    create_database(database_path, loaded.scenes, loaded.keyframes, meta)
    manifest = {**meta, 'runtime': {'python': sys.version.split()[0], 'platform': platform.platform(), 'numpy': np.__version__}, 'files': {'database': database_path.name}, 'scene_vector_index': scene_vector_manifest, 'frame_vector_index': frame_vector_manifest}
    manifest_path = index_dir / 'index_manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return BuildReport(index_dir=str(index_dir), database_path=str(database_path), vector_backend=scene_vector_manifest['backend'] + (f"+{frame_vector_manifest['backend']}" if frame_vector_manifest else ''), embedding_dimension=loaded.embedding_dimension, scene_count=len(loaded.scenes), keyframe_count=len(loaded.keyframes), video_count=len({scene.video_id for scene in loaded.scenes}), warnings=loaded.warnings, elapsed_sec=round(time.perf_counter() - started, 4))

## 8. Search engine class

Đây là lớp chạy chính:

- `search()` chạy các nhánh song song, lọc candidate, gộp RRF và trả evidence;
- `search_sequence()` gọi `search()` cho từng bước rồi dùng beam search để
  kiểm tra đúng video, đúng thứ tự scene/event và giới hạn khoảng thời gian.

In [81]:
from __future__ import annotations

# ===========================================================================
# Logic preserved from: engine.py
# ===========================================================================
import json
from pathlib import Path
from typing import Any
import numpy as np

class LocalHybridSearchEngine:
    """Local scene/frame retrieval with six parallel branches and RRF."""

    def __init__(self, index_dir: str | Path, device: str | None=None, asset_root: str | Path | None=None):
        self.index_dir = Path(index_dir).expanduser().resolve()
        manifest_path = self.index_dir / 'index_manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'Missing index manifest: {manifest_path}')
        self.manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if int(self.manifest.get('schema_version', 0)) != 2:
            raise ValueError('This engine expects search index schema_version=2; rebuild the index.')
        self.config = EngineConfig.from_dict(self.manifest.get('config', {}))
        self.connection = connect_database(self.index_dir / self.manifest['files']['database'], readonly=True)
        self.scene_vector_index = VectorIndex(self.index_dir, self.manifest['scene_vector_index'])
        frame_manifest = self.manifest.get('frame_vector_index')
        self.frame_vector_index = VectorIndex(self.index_dir, frame_manifest) if frame_manifest else None
        self.embedding_model = str(self.manifest['scene_embedding_model'])
        self.device = device
        self.asset_root = Path(asset_root).expanduser().resolve() if asset_root else None
        self._text_encoder: OpenClipTextEncoder | None = None
        self._text_encoder_error: str | None = None

    def close(self) -> None:
        self.connection.close()

    def __enter__(self) -> 'LocalHybridSearchEngine':
        return self

    def __exit__(self, exc_type, exc, traceback) -> None:
        self.close()

    def encode_visual_query(self, text: str) -> np.ndarray:
        if self._text_encoder_error is not None:
            raise RuntimeError(self._text_encoder_error)
        if self._text_encoder is None:
            try:
                self._text_encoder = OpenClipTextEncoder(self.embedding_model, self.device)
            except RuntimeError as exc:
                self._text_encoder_error = str(exc)
                raise
        return self._text_encoder.encode(text)

    def _asset_path(self, value: str) -> str:
        if not value or Path(value).is_absolute() or self.asset_root is None:
            return value
        direct = self.asset_root / value
        if direct.exists():
            return str(direct.resolve())
        matches = list(self.asset_root.rglob(Path(value).name))
        return str(matches[0].resolve()) if len(matches) == 1 else value

    def _resolve_frame(self, frame: dict | None) -> dict | None:
        if frame is None:
            return None
        output = dict(frame)
        output['image_path'] = self._asset_path(output.get('image_path', ''))
        return output

    def _scene_vector_candidates(self, query_vector: np.ndarray, limit: int, video_id: str | None, start_sec: float | None, end_sec: float | None) -> list[dict]:
        hits = self.scene_vector_index.search(query_vector, max(limit * 2, limit))
        scenes = fetch_scenes_by_vector_rows(self.connection, [row for row, _ in hits])
        output = []
        for row, score in hits:
            if float(score) < self.config.min_scene_vector_similarity:
                continue
            scene = scenes.get(row)
            if scene is None:
                continue
            if self.config.exclude_invalid and scene['quality_status'] == 'invalid':
                continue
            if video_id and scene['video_id'] != video_id:
                continue
            if start_sec is not None and scene['end_sec'] < start_sec:
                continue
            if end_sec is not None and scene['start_sec'] > end_sec:
                continue
            output.append({'scene_id': scene['scene_id'], 'branch': 'scene_vector', 'score': float(score)})
            if len(output) >= limit:
                break
        return output

    def _frame_vector_candidates(self, query_vector: np.ndarray, limit: int, video_id: str | None, start_sec: float | None, end_sec: float | None) -> list[dict]:
        if self.frame_vector_index is None:
            return []
        hits = self.frame_vector_index.search(query_vector, max(limit * 5, limit))
        frames = fetch_frames_by_vector_rows(self.connection, [row for row, _ in hits])
        scene_ids = list({frame['scene_id'] for frame in frames.values()})
        scenes = fetch_scenes(self.connection, scene_ids)
        best: dict[str, dict] = {}
        for row, score in hits:
            if float(score) < self.config.min_frame_vector_similarity:
                continue
            frame = frames.get(row)
            if frame is None:
                continue
            scene = scenes.get(frame['scene_id'])
            if scene is None:
                continue
            if self.config.exclude_invalid and scene['quality_status'] == 'invalid':
                continue
            if video_id and scene['video_id'] != video_id:
                continue
            if start_sec is not None and scene['end_sec'] < start_sec:
                continue
            if end_sec is not None and scene['start_sec'] > end_sec:
                continue
            previous = best.get(frame['scene_id'])
            if previous is None or score > previous['score']:
                best[frame['scene_id']] = {'scene_id': frame['scene_id'], 'branch': 'frame_vector', 'score': float(score), 'best_frame': frame}
        return sorted(best.values(), key=lambda item: -item['score'])[:limit]

    def search(self, text_query: str, *, visual_query: str | None=None, query_vector: np.ndarray | None=None, use_vector: bool=True, task: str='auto', top_k: int=10, video_id: str | None=None, start_sec: float | None=None, end_sec: float | None=None, match_all_terms: bool=False) -> list[dict[str, Any]]:
        if top_k <= 0:
            return []
        plan = plan_query(text_query, self.config, task=task)
        branch_results: dict[str, list[dict]] = {}
        for branch in ('semantic', 'ocr', 'speech', 'tags'):
            branch_results[branch] = search_branch(self.connection, branch, text_query, self.config.lexical_candidates, self.config, video_id=video_id, start_sec=start_sec, end_sec=end_sec, match_all=match_all_terms)
        branch_results['event'] = search_event_branch(self.connection, text_query, self.config.lexical_candidates, self.config, video_id=video_id, start_sec=start_sec, end_sec=end_sec, match_all=match_all_terms)
        vector_status = 'disabled'
        if use_vector:
            vector_status = 'requested'
            if query_vector is None:
                vector_text = (visual_query or '').strip()
                if not vector_text and (not self.config.require_visual_query_for_vector):
                    vector_text = text_query.strip()
                if not vector_text and self.config.require_visual_query_for_vector:
                    vector_status = 'skipped: visual_query_required'
                if vector_text.strip():
                    try:
                        query_vector = self.encode_visual_query(vector_text)
                    except RuntimeError as exc:
                        vector_status = 'skipped: ' + str(exc).splitlines()[0]
            if query_vector is not None:
                vector_status = 'used'
                branch_results['scene_vector'] = self._scene_vector_candidates(query_vector, self.config.vector_candidates, video_id, start_sec, end_sec)
                branch_results['frame_vector'] = self._frame_vector_candidates(query_vector, self.config.vector_candidates, video_id, start_sec, end_sec)
        branch_results.setdefault('scene_vector', [])
        branch_results.setdefault('frame_vector', [])
        ranked_lists: list[tuple[list[str], float]] = []
        all_scene_ids: set[str] = set()
        for branch, results in branch_results.items():
            ids = [item['scene_id'] for item in results]
            all_scene_ids.update(ids)
            if ids and plan.branch_weights.get(branch, 0.0) > 0:
                ranked_lists.append((ids, plan.branch_weights[branch]))
        scenes = fetch_scenes(self.connection, sorted(all_scene_ids))
        multipliers = {scene_id: float(scene['quality_penalty']) for scene_id, scene in scenes.items()}
        fused = reciprocal_rank_fusion(ranked_lists, self.config.rrf_k, item_multipliers=multipliers)[:top_k]
        if not fused:
            return []
        by_branch = {branch: {item['scene_id']: item for item in results} for branch, results in branch_results.items()}
        ranks = {branch: {item['scene_id']: rank for rank, item in enumerate(results, 1)} for branch, results in branch_results.items()}
        output: list[dict[str, Any]] = []
        for final_rank, (scene_id, rrf_score) in enumerate(fused, 1):
            scene = scenes[scene_id]
            frame_item = by_branch['frame_vector'].get(scene_id, {})
            best_frame = frame_item.get('best_frame') or representative_frame(self.connection, scene_id)
            branch_ranks = {branch: branch_rank[scene_id] for branch, branch_rank in ranks.items() if scene_id in branch_rank}
            branch_scores = {branch: by_branch[branch][scene_id]['score'] for branch in by_branch if scene_id in by_branch[branch]}
            branch_coverages = {branch: float(by_branch[branch][scene_id].get('lexical_coverage', 0.0)) for branch in ('semantic', 'ocr', 'speech', 'tags', 'event') if scene_id in by_branch[branch]}
            matched_terms = sorted({term for branch in ('semantic', 'ocr', 'speech', 'tags', 'event') if scene_id in by_branch[branch] for term in by_branch[branch][scene_id].get('matched_terms', [])})
            snippets = {branch: by_branch[branch][scene_id].get('snippet', '') for branch in ('semantic', 'ocr', 'speech', 'tags', 'event') if scene_id in by_branch[branch] and by_branch[branch][scene_id].get('snippet')}
            matched_event = by_branch['event'].get(scene_id, {}).get('matched_event')
            lexical_ranks = [rank for branch, rank in branch_ranks.items() if branch not in {'scene_vector', 'frame_vector'}]
            vector_ranks = [rank for branch, rank in branch_ranks.items() if branch in {'scene_vector', 'frame_vector'}]
            output.append({'rank': final_rank, 'scene_id': scene_id, 'video_id': scene['video_id'], 'scene_no': scene['scene_no'], 'start_sec': scene['start_sec'], 'end_sec': scene['end_sec'], 'rrf_score': float(rrf_score), 'query_plan': {'task': plan.task, 'hints': plan.hints, 'vector_status': vector_status}, 'branch_ranks': branch_ranks, 'branch_scores': branch_scores, 'branch_coverages': branch_coverages, 'lexical_coverage': max(branch_coverages.values(), default=0.0), 'matched_terms': matched_terms, 'lexical_rank': min(lexical_ranks) if lexical_ranks else None, 'vector_rank': min(vector_ranks) if vector_ranks else None, 'snippets': snippets, 'matched_event': matched_event, 'caption_vi': scene['caption_vi'], 'caption_en': scene['caption_en'], 'ocr_text': scene['ocr_text'], 'transcript': scene['transcript'], 'keywords': scene['keywords'], 'entities': scene['entities'], 'actions': scene['actions'], 'quality_status': scene['quality_status'], 'quality_penalty': scene['quality_penalty'], 'quality_errors': scene['quality_errors'], 'clip_path': self._asset_path(scene['clip_path']), 'best_frame': self._resolve_frame(best_frame)})
        return output

    def search_sequence(self, steps: list[str], *, per_step_k: int=30, top_k: int=5, max_gap_sec: float | None=120.0, use_vector: bool=True, visual_steps: list[str] | None=None, beam_width: int=200) -> list[dict[str, Any]]:
        if len(steps) < 2:
            raise ValueError('Temporal search needs at least two ordered steps')
        if visual_steps is not None and len(visual_steps) != len(steps):
            raise ValueError('visual_steps must have the same length as steps')
        candidates = [self.search(step, visual_query=visual_steps[index] if visual_steps else None, use_vector=use_vector, task='temporal', top_k=per_step_k) for index, step in enumerate(steps)]
        if any((not group for group in candidates)):
            return []

        def anchor(item: dict) -> tuple[float, int | None]:
            event = item.get('matched_event')
            if event:
                return (float(event['absolute_start_sec']), int(event['event_order']))
            return (float(item['start_sec']), None)
        beams: list[tuple[float, list[dict]]] = [(item['rrf_score'], [item]) for item in candidates[0]]
        for group in candidates[1:]:
            next_beams: list[tuple[float, list[dict]]] = []
            for score, sequence in beams:
                previous = sequence[-1]
                previous_time, previous_event_order = anchor(previous)
                for item in group:
                    if item['video_id'] != previous['video_id']:
                        continue
                    current_time, current_event_order = anchor(item)
                    same_scene_event_order = item['scene_id'] == previous['scene_id'] and previous_event_order is not None and (current_event_order is not None) and (current_event_order > previous_event_order)
                    later_scene = item['scene_no'] > previous['scene_no']
                    if not (same_scene_event_order or later_scene):
                        continue
                    gap = max(0.0, current_time - previous_time)
                    if max_gap_sec is not None and gap > max_gap_sec:
                        continue
                    next_beams.append((score + item['rrf_score'] - gap * 0.0001, [*sequence, item]))
            beams = sorted(next_beams, key=lambda pair: -pair[0])[:beam_width]
            if not beams:
                return []
        return [{'rank': rank, 'score': float(score), 'video_id': sequence[0]['video_id'], 'start_sec': anchor(sequence[0])[0], 'end_sec': anchor(sequence[-1])[0], 'scene_ids': [item['scene_id'] for item in sequence], 'steps': sequence} for rank, (score, sequence) in enumerate(beams[:top_k], 1)]

## 9. Hàm notebook thay cho CLI

File Python cũ có CLI `build`, `inspect`, `query`, `sequence`. Notebook gọi
trực tiếp các class/hàm tương ứng nên không cần `argparse`.

In [82]:
from pathlib import Path
from typing import Any
import json
import numpy as np


def inspect_index(index_dir: str | Path) -> dict[str, Any]:
    path = Path(index_dir).expanduser().resolve() / "index_manifest.json"
    if not path.exists():
        raise FileNotFoundError(f"Chưa có index manifest: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def search_index(
    index_dir: str | Path,
    text: str,
    *,
    visual_text: str | None = None,
    query_vector: np.ndarray | None = None,
    use_vector: bool = False,
    task: str = "scene",
    top_k: int = 5,
    video_id: str | None = None,
    start_sec: float | None = None,
    end_sec: float | None = None,
    match_all_terms: bool = False,
    asset_root: str | Path | None = None,
) -> list[dict[str, Any]]:
    with LocalHybridSearchEngine(index_dir, asset_root=asset_root) as engine:
        return engine.search(
            text,
            visual_query=visual_text,
            query_vector=query_vector,
            use_vector=use_vector,
            task=task,
            top_k=top_k,
            video_id=video_id,
            start_sec=start_sec,
            end_sec=end_sec,
            match_all_terms=match_all_terms,
        )


def sequence_index(
    index_dir: str | Path,
    steps: list[str],
    *,
    use_vector: bool = False,
    visual_steps: list[str] | None = None,
    per_step_k: int = 30,
    top_k: int = 5,
    max_gap_sec: float | None = 120.0,
    beam_width: int = 200,
) -> list[dict[str, Any]]:
    with LocalHybridSearchEngine(index_dir) as engine:
        return engine.search_sequence(
            steps,
            use_vector=use_vector,
            visual_steps=visual_steps,
            per_step_k=per_step_k,
            top_k=top_k,
            max_gap_sec=max_gap_sec,
            beam_width=beam_width,
        )


def compact_hits(hits: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [
        {
            "rank": hit["rank"],
            "scene_id": hit["scene_id"],
            "time": [round(hit["start_sec"], 3), round(hit["end_sec"], 3)],
            "branches": hit["branch_ranks"],
            "coverage": round(hit["lexical_coverage"], 3),
            "rrf": round(hit["rrf_score"], 6),
        }
        for hit in hits
    ]

## 10. Cấu hình input/output

Trên Kaggle, `INPUT_ROOT=/kaggle/input` và
`INDEX_DIR=/kaggle/working/08_local_search_index`. Chạy local thì sửa hai
đường dẫn dưới đây hoặc đặt biến môi trường `AIC_INPUT_ROOT`,
`AIC_INDEX_DIR`.

In [83]:
from pathlib import Path
import os

IS_KAGGLE = Path("/kaggle/input").exists()
DEFAULT_INPUT = Path("/kaggle/input") if IS_KAGGLE else Path("./component_outputs")
DEFAULT_INDEX = (
    Path("/kaggle/working/08_local_search_index")
    if IS_KAGGLE
    else Path("./08_local_search_index")
)

INPUT_ROOT = Path(os.environ.get("AIC_INPUT_ROOT", DEFAULT_INPUT)).expanduser().resolve()
INDEX_DIR = Path(os.environ.get("AIC_INDEX_DIR", DEFAULT_INDEX)).expanduser().resolve()
VECTOR_BACKEND = os.environ.get("AIC_VECTOR_BACKEND", "auto")

print("INPUT_ROOT =", INPUT_ROOT)
print("INDEX_DIR  =", INDEX_DIR)
print("VECTOR_BACKEND =", VECTOR_BACKEND)

INPUT_ROOT = /kaggle/input
INDEX_DIR  = /kaggle/working/08_local_search_index
VECTOR_BACKEND = auto


## 11. Build index

In [84]:
if not INPUT_ROOT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy INPUT_ROOT={INPUT_ROOT}. "
        "Hãy sửa đường dẫn ở cell cấu hình."
    )

config = EngineConfig(
    vector_backend=VECTOR_BACKEND,
    lexical_candidates=100,
    vector_candidates=100,
    min_lexical_terms=2,
    min_lexical_coverage=0.34,
    require_visual_query_for_vector=True,
    needs_review_penalty=0.75,
    exclude_invalid=True,
)

build_report = build_index(INPUT_ROOT, INDEX_DIR, config)
print(json.dumps(build_report.to_dict(), ensure_ascii=False, indent=2))

{
  "index_dir": "/kaggle/working/08_local_search_index",
  "database_path": "/kaggle/working/08_local_search_index/aic_search.db",
  "vector_backend": "faiss_hnsw+faiss_hnsw",
  "embedding_dimension": 512,
  "scene_count": 10,
  "keyframe_count": 33,
  "video_count": 1,
  "warnings": [],
  "elapsed_sec": 1.3646
}


## 12. Inspect index

In [85]:
manifest = inspect_index(INDEX_DIR)
index_summary = {
    "schema_version": manifest["schema_version"],
    "stats": manifest["stats"],
    "scene_embedding_model": manifest["scene_embedding_model"],
    "scene_vector_index": manifest["scene_vector_index"],
    "frame_vector_index": manifest.get("frame_vector_index"),
    "warnings": manifest["warnings"],
}
print(json.dumps(index_summary, ensure_ascii=False, indent=2))

{
  "schema_version": 2,
  "stats": {
    "source_roots": [
      "/kaggle/input"
    ],
    "scene_count": 10,
    "keyframe_count": 33,
    "video_count": 1,
    "semantic_scene_count": 10,
    "validation_scene_count": 8,
    "validation_report_count": 2,
    "validation_passed": true,
    "passed_scene_count": 2,
    "needs_review_scene_count": 8,
    "invalid_scene_count": 0
  },
  "scene_embedding_model": "open_clip:ViT-B-32:openai",
  "scene_vector_index": {
    "name": "scene",
    "backend": "faiss_hnsw",
    "count": 10,
    "dimension": 512,
    "metric": "cosine_via_normalized_inner_product",
    "numpy_file": null,
    "faiss_file": "scene_hnsw.faiss",
    "hnsw": {
      "m": 32,
      "ef_construction": 200,
      "ef_search": 64
    }
  },
  "frame_vector_index": {
    "name": "frame",
    "backend": "faiss_hnsw",
    "count": 33,
    "dimension": 512,
    "metric": "cosine_via_normalized_inner_product",
    "numpy_file": null,
    "faiss_file": "frame_hnsw.faiss",
    

## 13. Text search

`use_vector=False` chỉ chạy FTS5/BM25 + filter + weighted RRF, phù hợp khi
query tiếng Việt và chưa cài OpenCLIP.

In [86]:
TEXT_QUERY = "Bộ Công an vào cuộc vụ thịt heo bẩn"

text_hits = search_index(
    INDEX_DIR,
    TEXT_QUERY,
    use_vector=False,
    task="scene",
    top_k=5,
    asset_root=INPUT_ROOT,
)

print(json.dumps(compact_hits(text_hits), ensure_ascii=False, indent=2))

[
  {
    "rank": 1,
    "scene_id": "K16_V001_S0005",
    "time": [
      16.3,
      17.633
    ],
    "branches": {
      "semantic": 1,
      "ocr": 1,
      "speech": 3
    },
    "coverage": 0.875,
    "rrf": 0.044118
  },
  {
    "rank": 2,
    "scene_id": "K16_V001_S0007",
    "time": [
      18.933,
      19.933
    ],
    "branches": {
      "speech": 5
    },
    "coverage": 0.875,
    "rrf": 0.015385
  },
  {
    "rank": 3,
    "scene_id": "K16_V001_S0002",
    "time": [
      0.9,
      12.7
    ],
    "branches": {
      "speech": 6
    },
    "coverage": 0.875,
    "rrf": 0.015152
  },
  {
    "rank": 4,
    "scene_id": "K16_V001_S0003",
    "time": [
      12.733,
      14.867
    ],
    "branches": {
      "speech": 1
    },
    "coverage": 0.875,
    "rrf": 0.012295
  },
  {
    "rank": 5,
    "scene_id": "K16_V001_S0004",
    "time": [
      14.9,
      16.267
    ],
    "branches": {
      "speech": 2
    },
    "coverage": 0.875,
    "rrf": 0.012097
  }
]


## 14. Vector/hybrid search tùy chọn

Có hai cách:

1. truyền `query_vector` đã được tạo bởi đúng model OpenCLIP của index;
2. truyền `visual_text` bằng tiếng Anh và bật `AIC_INSTALL_OPENCLIP=1` trước
   khi chạy cell cài đặt.

Cell dưới đây mặc định không chạy model.

In [87]:
RUN_VISUAL_EXAMPLE = False

if RUN_VISUAL_EXAMPLE:
    visual_hits = search_index(
        INDEX_DIR,
        text="cửa hàng bán thịt heo",
        visual_text="a yellow pork shop storefront",
        use_vector=True,
        task="scene",
        top_k=5,
        asset_root=INPUT_ROOT,
    )
    print(json.dumps(compact_hits(visual_hits), ensure_ascii=False, indent=2))
else:
    print("Bỏ qua visual example. Đặt RUN_VISUAL_EXAMPLE=True để chạy.")

Bỏ qua visual example. Đặt RUN_VISUAL_EXAMPLE=True để chạy.


## 15. Temporal sequence search

In [88]:
SEQUENCE_STEPS = [
    "Bộ Công an vào cuộc vụ thịt heo bẩn",
    "Lạng Sơn nhận bằng UNESCO",
]

sequence_hits = sequence_index(
    INDEX_DIR,
    SEQUENCE_STEPS,
    use_vector=False,
    per_step_k=30,
    top_k=5,
    max_gap_sec=180.0,
)

sequence_summary = [
    {
        "rank": item["rank"],
        "video_id": item["video_id"],
        "scene_ids": item["scene_ids"],
        "score": round(item["score"], 6),
    }
    for item in sequence_hits
]
print(json.dumps(sequence_summary, ensure_ascii=False, indent=2))

[
  {
    "rank": 1,
    "video_id": "K16_V001",
    "scene_ids": [
      "K16_V001_S0005",
      "K16_V001_S0009"
    ],
    "score": 0.097999
  },
  {
    "rank": 2,
    "video_id": "K16_V001",
    "scene_ids": [
      "K16_V001_S0005",
      "K16_V001_S0008"
    ],
    "score": 0.097331
  },
  {
    "rank": 3,
    "video_id": "K16_V001",
    "scene_ids": [
      "K16_V001_S0005",
      "K16_V001_S0010"
    ],
    "score": 0.081028
  },
  {
    "rank": 4,
    "video_id": "K16_V001",
    "scene_ids": [
      "K16_V001_S0007",
      "K16_V001_S0009"
    ],
    "score": 0.072849
  },
  {
    "rank": 5,
    "video_id": "K16_V001",
    "scene_ids": [
      "K16_V001_S0007",
      "K16_V001_S0008"
    ],
    "score": 0.072181
  }
]


## 16. Smoke test

In [89]:
assert build_report.scene_count > 0
assert manifest["schema_version"] == 2
assert len(manifest["stats"]["source_roots"]) >= 1
assert all(hit["rank"] == index for index, hit in enumerate(text_hits, 1))

# Kiểm tra hồi quy chỉ kích hoạt khi đúng bộ mẫu K16_V001 10 scene.
sample_ids = [f"K16_V001_S{number:04d}" for number in range(1, 11)]
with LocalHybridSearchEngine(INDEX_DIR) as smoke_engine:
    indexed_ids = {
        row[0]
        for row in smoke_engine.connection.execute(
            "SELECT scene_id FROM scenes"
        ).fetchall()
    }
is_reference_4zip_sample = (
    set(sample_ids) == indexed_ids
    and manifest["stats"]["scene_count"] == 10
    and manifest["stats"]["video_count"] == 1
    and manifest["stats"]["semantic_scene_count"] == 0
)
if is_reference_4zip_sample:
    expected = [
        "K16_V001_S0005",
        "K16_V001_S0003",
        "K16_V001_S0004",
        "K16_V001_S0006",
        "K16_V001_S0007",
    ]
    actual = [hit["scene_id"] for hit in text_hits]
    assert actual == expected, {"expected": expected, "actual": actual}
    print("Regression ranking K16_V001: PASSED")

print("Notebook smoke test: PASSED")

Notebook smoke test: PASSED


## 17. Đóng gói index (tùy chọn)

Notebook không cần build lại mỗi lần search. Có thể ZIP index để tải từ Kaggle
và dùng lại trên máy local.

In [90]:
import shutil

EXPORT_INDEX_ZIP = True
if EXPORT_INDEX_ZIP:
    zip_base = (
        Path("/kaggle/working/08_local_search_index")
        if IS_KAGGLE
        else INDEX_DIR.parent / "08_local_search_index"
    )
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=INDEX_DIR))
    print("Đã tạo:", zip_path, f"({zip_path.stat().st_size / 1024**2:.2f} MiB)")
else:
    print("Đặt EXPORT_INDEX_ZIP=True nếu muốn đóng gói index.")

Đã tạo: /kaggle/working/08_local_search_index.zip (0.12 MiB)
